# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [2]:
# Dependencias
import re, io, tarfile, zipfile, math
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp
import math
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
import zstandard 


In [3]:
# Recoleccion del corpus

CORPUS_PATH = "txt-files.tar.zip"   
N_DOCS      = 1000                  
MIN_CHARS   = 5_000     # descartar documentos cortos            

# 
START_RE = re.compile(r"\*\*\* ?START OF (THE|THIS) PROJECT GUTENBERG.*?\*\*\*", re.I | re.S)
END_RE   = re.compile(r"\*\*\* ?END OF (THE|THIS) PROJECT GUTENBERG", re.I)
TITLE_RE = re.compile(r"^Title:\s*(.+)$", re.M)

def limpiar_gutenberg(raw):
    m = START_RE.search(raw)
    if m: raw = raw[m.end():]
    m = END_RE.search(raw)
    if m: raw = raw[:m.start()]
    return raw.strip()

def iterar_docs(path):
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            nombres = z.namelist()
            tars = [n for n in nombres if n.lower().endswith((".tar", ".tar.gz", ".tgz"))]
            if tars:
                # zip que contiene un tar: se lee el tar en streaming
                with z.open(tars[0]) as inner:
                    modo = "r|gz" if tars[0].lower().endswith((".gz", ".tgz")) else "r|"
                    tar = tarfile.open(fileobj=inner, mode=modo)
                    for member in tar:
                        if member.isfile() and member.name.lower().endswith(".txt"):
                            f = tar.extractfile(member)
                            yield member.name, f.read().decode("utf-8", errors="ignore")
            else:
                for name in nombres:
                    if name.lower().endswith(".txt"):
                        yield name, z.read(name).decode("utf-8", errors="ignore")
    else:
        if path.endswith(".zst"):
            fobj = zstandard.ZstdDecompressor().stream_reader(open(path, "rb"))
            tar = tarfile.open(fileobj=fobj, mode="r|")
        elif path.endswith((".gz", ".tgz")):
            tar = tarfile.open(path, mode="r|gz")
        else:
            tar = tarfile.open(path, mode="r|")
        for member in tar:
            if member.isfile() and member.name.lower().endswith(".txt"):
                f = tar.extractfile(member)
                yield member.name, f.read().decode("utf-8", errors="ignore")

In [4]:
# Tokenizacion
TOKEN_RE = re.compile(r"[a-z]+")
STOP = set(ENGLISH_STOP_WORDS)

def tokenizar(text):
    tokens = TOKEN_RE.findall(text.lower())
    return [t for t in tokens if len(t) > 2 and t not in STOP]


doc_ids, titulos, textos, docs = [], [], [], []
for name, raw in iterar_docs(CORPUS_PATH):
    if "Language: English" not in raw[:5000]:     # solo documentos en ingles
        continue
    t = TITLE_RE.search(raw[:5000])
    body = limpiar_gutenberg(raw)
    if len(body) < MIN_CHARS:
        continue
    doc_ids.append(f"D{len(docs):04d}")
    titulos.append(t.group(1).strip() if t else name)
    textos.append(body)
    docs.append(tokenizar(body))
    if len(docs) >= N_DOCS:
        break

N = len(docs)
vocab = sorted(set(t for d in docs for t in d))
lengths = np.array([len(d) for d in docs])

table = pd.DataFrame({
    "Identificador": doc_ids,
    "Título": titulos,
    "Términos únicos": [len(set(d)) for d in docs],
})
display(table.head(15))

,Identificador,Título,Términos únicos
0,D0000,The Magna Carta,1162
1,D0001,Apocolocyntosis,1360
2,D0002,The House on the Borderland,4658
3,D0003,"My First Years as a Frenchwoman, 1876-1879",5589
4,D0004,The Warriors,6279
5,D0005,A Voyage to the Moon,7269
6,D0006,La Fiammetta,2602
7,D0007,Carmilla,3736
8,D0008,The Mystery,7240
9,D0009,"Wild Northern Scenes; Or, Sporting Adventures ...",7292


In [5]:
print(f"Número de documentos : {N}")
print(f"Tamaño del vocabulario: {len(vocab):,}")

Número de documentos : 1000
Tamaño del vocabulario: 339,287


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [6]:
def calcular_tf(docs):
    tfs = []
    for tokens in docs:
        tf = {}
        for t in tokens:
            tf[t] = tf.get(t, 0) + 1
        tfs.append(tf)
    return tfs

def calcular_df(tfs):
    df = {}
    for tf in tfs:
        for t in tf:                      
            df[t] = df.get(t, 0) + 1
    return df

def calcular_idf(df, n_docs):
    return {t: math.log(n_docs / d) + 1.0 for t, d in df.items()}

def calcular_tfidf(tfs, idf):
    return [{t: f * idf[t] for t, f in tf.items()} for tf in tfs]

### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [7]:
# Vocabulario ordenado
vocab = sorted(set(t for d in docs for t in d))
term2idx = {t: i for i, t in enumerate(vocab)}
V = len(vocab)

# Calculo de TF
tfs = calcular_tf(docs)

# Calculo DF
df = calcular_df(tfs)

# Calculo IDF
idf = calcular_idf(df, N)
idf_vec = np.array([idf[t] for t in vocab])      

# TF-IDF
pesos = calcular_tfidf(tfs, idf)

filas, columnas, valores = [], [], []
for i, w in enumerate(pesos):
    for t, peso in w.items():
        filas.append(i)
        columnas.append(term2idx[t])
        valores.append(peso)

X_manual = sp.csr_matrix((valores, (filas, columnas)), shape=(N, V), dtype=np.float64)
print(f"Dimension de la matriz : {X_manual.shape[0]}  x {X_manual.shape[1]:,} ")
print(f"Entradas no nulas    : {X_manual.nnz:,}")

Dimension de la matriz : 1000  x 339,287 
Entradas no nulas    : 5,934,807


In [8]:
# Muestra de la matriz

n_docs_vis, n_terms_vis = 10, 15

sub = X_manual[:n_docs_vis]
suma_terminos = np.asarray(sub.sum(axis=0)).ravel()
top_cols = np.argsort(-suma_terminos)[:n_terms_vis]

df_sub = pd.DataFrame(
    sub[:, top_cols].toarray(),
    index=doc_ids[:n_docs_vis],
    columns=[vocab[j] for j in top_cols],
)
display(df_sub.round(2))

,darrow,trendon,said,man,great,time,brahmin,little,men,like,shall,barnett,thrackles,day,old
D0000,0.00,0.00,23.32,27.35,3.05,20.08,0.00,1.00,39.24,9.05,334.92,0.00,0.0,22.24,7.09
D0001,0.00,0.00,12.17,15.20,10.15,4.02,0.00,3.01,9.29,9.05,3.08,0.00,0.0,13.14,3.04
D0002,0.00,0.00,42.59,23.30,146.18,161.65,0.00,101.20,1.03,51.26,11.30,0.00,0.0,55.61,68.89
D0003,0.00,0.00,129.80,90.16,174.60,96.38,0.00,107.21,118.74,60.30,1.03,0.00,0.0,117.28,97.26
D0004,0.00,0.00,15.21,297.85,152.27,73.29,0.00,29.06,199.28,34.17,142.80,0.00,0.0,133.46,11.14
D0005,0.00,0.00,134.88,119.54,131.96,147.59,776.49,86.17,49.56,71.36,21.57,0.00,0.0,54.60,31.41
D0006,0.00,0.00,10.14,7.09,16.24,31.12,0.00,8.02,15.49,14.07,3.08,0.00,0.0,17.19,9.12
D0007,0.00,0.00,115.61,18.24,35.53,65.26,0.00,74.15,7.23,41.21,39.04,0.00,0.0,23.25,53.69
D0008,1118.26,1107.09,362.03,176.28,52.79,121.48,0.00,132.26,159.01,145.73,35.96,625.99,616.8,54.60,70.92
D0009,0.00,0.00,206.88,135.75,170.54,147.59,0.00,209.42,73.31,216.08,46.23,0.00,0.0,110.21,244.15


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [9]:
# creacion del vectorizer
vectorizer = TfidfVectorizer(analyzer=lambda tokens: tokens, use_idf=True, smooth_idf=False, sublinear_tf=False, norm=None,)
X_sk = vectorizer.fit_transform(docs).tocsr()

print(f"Dimension de la matriz TfidfVectorizer: {X_sk.shape[0]} x {X_sk.shape[1]:,}")
print(f"Entradas no nulas    : {X_sk.nnz:,}")

Dimension de la matriz TfidfVectorizer: 1000 x 339,287
Entradas no nulas    : 5,934,807


In [10]:
# Verificacion de vocabulario
sk_vocab = list(vectorizer.get_feature_names_out())
print("Mantienen el mismo vocabulario y orden:", sk_vocab == vocab)

# Verificacion de IDF
print("IDF es similar en ambos:", np.allclose(idf_vec, vectorizer.idf_))
print(f"Máxima diferencia en IDF: {np.abs(idf_vec - vectorizer.idf_).max():.3e}")

# Maxima diferencia absoluta entre matrices
diferencia = abs(X_manual - X_sk)
max_diff = diferencia.max()
print(f"Máxima diferencia absoluta entre matrices: {max_diff:.3e}")
print("La matriz TF-IDF es similar:", max_diff < 1e-9)

Mantienen el mismo vocabulario y orden: True
IDF es similar en ambos: True
Máxima diferencia en IDF: 0.000e+00
Máxima diferencia absoluta entre matrices: 0.000e+00
La matriz TF-IDF es similar: True


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

## Consultas
Las consulta se hicieron en ingles ya que solo se extrajeron textos en ingles del corpus para mantener coherencia y evitar ruido.

1. love and marriage in English society
    - Relatos o narraciones románticas en la sociedad inglesa.
2. sea voyage ship captain storm
    - Relatos o cuentos acerca de viajes maritimos o peripecias en el altamar desde el punto de vista del capitán.
3. war battle soldiers army victory
    - Relatos o historias acerca de enfrentamientos en guerras o batallas históricas.
4. ghost haunted castle mystery murder
    - Relatos o leyendas sobrenaturales, crimenes o misterios

In [11]:
consultas = {
    "Q1": {"texto": "love and marriage in English society"},
    "Q2": {"texto": "sea voyage ship captain storm"},
    "Q3": {"texto": "war battle soldiers army victory"},
    "Q4": {"texto": "ghost haunted castle mystery murder"}
}

# Preproseamiento
for qid, q in consultas.items():
    q["tokens"] = tokenizar(q["texto"])                    
    q["vec"] = vectorizer.transform([q["tokens"]])         
    en_vocab = [t for t in q["tokens"] if t in term2idx]
    fuera = [t for t in q["tokens"] if t not in term2idx]
    print(f"{qid}: «{q['texto']}»")
    print(f"   Tokens          : {q['tokens']}")
    print(f"   Vocabulario de la consulta  : {en_vocab}")
    if fuera:
        print(f"   Fuera del vocab.: {fuera} (se ignoran)")

Q1: «love and marriage in English society»
   Tokens          : ['love', 'marriage', 'english', 'society']
   Vocabulario de la consulta  : ['love', 'marriage', 'english', 'society']
Q2: «sea voyage ship captain storm»
   Tokens          : ['sea', 'voyage', 'ship', 'captain', 'storm']
   Vocabulario de la consulta  : ['sea', 'voyage', 'ship', 'captain', 'storm']
Q3: «war battle soldiers army victory»
   Tokens          : ['war', 'battle', 'soldiers', 'army', 'victory']
   Vocabulario de la consulta  : ['war', 'battle', 'soldiers', 'army', 'victory']
Q4: «ghost haunted castle mystery murder»
   Tokens          : ['ghost', 'haunted', 'castle', 'mystery', 'murder']
   Vocabulario de la consulta  : ['ghost', 'haunted', 'castle', 'mystery', 'murder']


In [17]:
def vector_manual(tokens):
    tf_q = {}
    for t in tokens:
        if t in term2idx:                
            tf_q[t] = tf_q.get(t, 0) + 1
    return {t: f * idf[t] for t, f in tf_q.items()}

def vector_libreria(vec):
    return {vocab[j]: float(v) for j, v in zip(vec.indices, vec.data)}

for qid, q in consultas.items():
    q["w_manual"] = vector_manual(q["tokens"])
    q["w_lib"] = vector_libreria(q["vec"])

    print(f"{qid}: «{q['texto']}»")
    print("  Manual   :", {t: round(w, 4) for t, w in q["w_manual"].items()})
    print("  Librería :", {t: round(w, 4) for t, w in q["w_lib"].items()})


Q1: «love and marriage in English society»
  Manual   : {'love': 1.1087, 'marriage': 1.4764, 'english': 1.2485, 'society': 1.3175}
  Librería : {'english': 1.2485, 'love': 1.1087, 'marriage': 1.4764, 'society': 1.3175}
Q2: «sea voyage ship captain storm»
  Manual   : {'sea': 1.1404, 'voyage': 1.8486, 'ship': 1.3857, 'captain': 1.5158, 'storm': 1.3653}
  Librería : {'captain': 1.5158, 'sea': 1.1404, 'ship': 1.3857, 'storm': 1.3653, 'voyage': 1.8486}
Q3: «war battle soldiers army victory»
  Manual   : {'war': 1.2282, 'battle': 1.3653, 'soldiers': 1.6088, 'army': 1.4262, 'victory': 1.5465}
  Librería : {'army': 1.4262, 'battle': 1.3653, 'soldiers': 1.6088, 'victory': 1.5465, 'war': 1.2282}
Q4: «ghost haunted castle mystery murder»
  Manual   : {'ghost': 1.8007, 'haunted': 2.0244, 'castle': 1.821, 'mystery': 1.5242, 'murder': 1.5516}
  Librería : {'castle': 1.821, 'ghost': 1.8007, 'haunted': 2.0244, 'murder': 1.5516, 'mystery': 1.5242}


In [18]:
def sim_coseno(q_vec, D):
    productos = (D @ q_vec.T).toarray().ravel()                          
    magnitud_q = np.sqrt(q_vec.multiply(q_vec).sum())                       
    magnitud_d = np.sqrt(np.asarray(D.multiply(D).sum(axis=1)).ravel())    
    denominador = magnitud_q * magnitud_d
    sim = np.zeros_like(productos)
    np.divide(productos, denominador, out=sim, where=denominador > 0)                
    return sim

for qid, q in consultas.items():
    q["sim"] = sim_coseno(q["vec"], X_sk)
    q["sim_sk"] = cosine_similarity(q["vec"], X_sk).ravel()
    dif = np.abs(q["sim"] - q["sim_sk"]).max()
    print(f"{qid}: máxima diferencia (Manual vs Librería) = {dif:.3e}")

Q1: máxima diferencia (Manual vs Librería) = 7.515e-15
Q2: máxima diferencia (Manual vs Librería) = 4.080e-15
Q3: máxima diferencia (Manual vs Librería) = 2.692e-15
Q4: máxima diferencia (Manual vs Librería) = 1.082e-15


In [14]:
def verificar_similitud(qid, d_idx):
    q_vec, d_vec = consultas[qid]["vec"], X_sk[d_idx]
    q_arr, d_arr = q_vec.toarray().ravel(), d_vec.toarray().ravel()

    compartidos = np.intersect1d(q_vec.indices, d_vec.indices)   # columnas donde ambos son no nulos
    tabla = pd.DataFrame({
        "Término compartido": [vocab[j] for j in compartidos],
        "Producto parcial (q_i · d_i)": q_arr[compartidos] * d_arr[compartidos],
        "q_i": q_arr[compartidos],
        "d_i": d_arr[compartidos],
    })
    display(tabla)

    producto_punto = tabla["Producto parcial (q_i · d_i)"].sum()
    norma_q = np.sqrt((q_arr ** 2).sum())
    norma_d = np.sqrt((d_arr ** 2).sum())
    similitud = producto_punto / (norma_q * norma_d)

    print(f"Producto punto : {producto_punto:.4f}")
    print(f"||q||          : {norma_q:.4f}")
    print(f"||d||          : {norma_d:.4f}")
    print(f"Similitud      : {similitud:.6f}")
    print(f"sklearn        : {cosine_similarity(q_vec, d_vec)[0, 0]:.6f}")

qid = "Q1"
mejor = int(np.argmax(consultas[qid]["sim"]))
print(f"Consulta {qid} vs {doc_ids[mejor]} – {titulos[mejor]}\n")
verificar_similitud(qid, mejor)

Consulta Q1 vs D0064 – Happiness and Marriage



,Término compartido,Producto parcial (q_i · d_i),q_i,d_i
0,english,1.558656,1.248461,1.248461
1,love,130.296726,1.108699,117.522138
2,marriage,47.956225,1.476424,32.481332
3,society,8.678428,1.317454,6.587271


Producto punto : 188.4900
||q||          : 2.5891
||d||          : 488.6785
Similitud      : 0.148977
sklearn        : 0.148977


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [19]:
TOLERANCIA = 1e-9
pd.set_option("display.max_colwidth", 250)

# Calculo manual
for qid, q in consultas.items():
    q["sim_manual"] = sim_coseno(q["vec"], X_manual)

# Calculo de la liberia
for qid, q in consultas.items():
    q["sim_lib"] = cosine_similarity(q["vec"], X_sk).ravel()

def fragmento(texto, tokens_consulta, ancho=120):
    for t in tokens_consulta:
        m = re.search(r"\b" + re.escape(t) + r"\w*", texto, re.I)
        if m:
            ini, fin = max(0, m.start() - ancho), min(len(texto), m.end() + ancho)
            return "…" + re.sub(r"\s+", " ", texto[ini:fin]) + "…"
    return re.sub(r"\s+", " ", texto[:2 * ancho]) + "…"

# Creacion del ranking top 5
def ranking_top(qid, clave, k=5):
    q = consultas[qid]
    top = np.argsort(-q[clave], kind="stable")[:k]
    return top, pd.DataFrame({
        "Posición": range(1, k + 1),
        "Identificador": [doc_ids[i] for i in top],
        "Título": [titulos[i] for i in top],
        "Puntuación": [round(q[clave][i], 4) for i in top],
        "Fragmento": [fragmento(textos[i], q["tokens"]) for i in top],
    })

for qid, q in consultas.items():
    print("=" * 100)
    print(f"{qid}: «{q['texto']}»")

    _, tabla_manual = ranking_top(qid, "sim_manual")
    print("\n--- Ranking top-5: implementación MANUAL ---")
    display(tabla_manual)

    _, tabla_lib = ranking_top(qid, "sim_lib")
    print("--- Ranking top-5: LIBRERÍA ---")
    display(tabla_lib)

Q1: «love and marriage in English society»

--- Ranking top-5: implementación MANUAL ---


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,D0064,Happiness and Marriage,0.1490,…w of Attraction? I want to make our married life happy and peaceful. I long for a wedded life of pure blessedness and love and joy without even a pinhead of bitterness ever finding lodgment in our household. How can I attain this state of pe…
1,2,D0744,English Poems,0.1466,"…lienne EPISTLE DEDICATORY _Dear Sister: Hear the conclusion of the whole matter. You dream like mad, you love like tinder, you aspire like a star-struck moth--for what? That you may hive little lyrics, and sell to a publisher fo…"
2,3,D0383,"Beacon Lights of History, Volume 07: Great Women",0.1405,"…LORD, LL.D., AUTHOR OF ""THE OLD ROMAN WORLD,"" ""MODERN EUROPE,"" ETC., ETC. CONTENTS. HÉLOÏSE. LOVE. Love, the flower of Eden The two Venuses of Socrates The Venus Urania The memory of Héloïse cherished Her birth…"
3,4,D0460,English Literature,0.1347,"…d Roger Ascham's suggestion, made over three centuries ago and still pertinent, that ""'tis a poor way to make a child love study by beginning with the things which he naturally dislikes."" We have laid emphasis upon the delights of literature…"
4,5,D0581,The Way of Peace,0.1300,"…R OF MEDITATION THE TWO MASTERS, SELF AND TRUTH THE ACQUIREMENT OF SPIRITUAL POWER THE REALIZATION OF SELFLESS LOVE ENTERING INTO THE INFINITE SAINTS, SAGES, AND SAVIORS; THE LAW OF SERVICE THE REALIZATION OF PERFECT PEACE …"


--- Ranking top-5: LIBRERÍA ---


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,D0064,Happiness and Marriage,0.1490,…w of Attraction? I want to make our married life happy and peaceful. I long for a wedded life of pure blessedness and love and joy without even a pinhead of bitterness ever finding lodgment in our household. How can I attain this state of pe…
1,2,D0744,English Poems,0.1466,"…lienne EPISTLE DEDICATORY _Dear Sister: Hear the conclusion of the whole matter. You dream like mad, you love like tinder, you aspire like a star-struck moth--for what? That you may hive little lyrics, and sell to a publisher fo…"
2,3,D0383,"Beacon Lights of History, Volume 07: Great Women",0.1405,"…LORD, LL.D., AUTHOR OF ""THE OLD ROMAN WORLD,"" ""MODERN EUROPE,"" ETC., ETC. CONTENTS. HÉLOÏSE. LOVE. Love, the flower of Eden The two Venuses of Socrates The Venus Urania The memory of Héloïse cherished Her birth…"
3,4,D0460,English Literature,0.1347,"…d Roger Ascham's suggestion, made over three centuries ago and still pertinent, that ""'tis a poor way to make a child love study by beginning with the things which he naturally dislikes."" We have laid emphasis upon the delights of literature…"
4,5,D0581,The Way of Peace,0.1300,"…R OF MEDITATION THE TWO MASTERS, SELF AND TRUTH THE ACQUIREMENT OF SPIRITUAL POWER THE REALIZATION OF SELFLESS LOVE ENTERING INTO THE INFINITE SAINTS, SAGES, AND SAVIORS; THE LAW OF SERVICE THE REALIZATION OF PERFECT PEACE …"


Q2: «sea voyage ship captain storm»

--- Ranking top-5: implementación MANUAL ---


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,D0439,The Mirror of the Sea,0.2871,"…07 Methuen & Co. edition by David Price, email ccx074@pglaf.org THE MIRROR OF THE SEA MEMORIES AND IMPRESSIONS BY …"
1,2,D0928,"Jack Mason, the Old Sailor",0.2372,"…ITH ILLUSTRATIONS. FOR CHILDREN. BY THEODORE THINKER. 1850. THE OLD SAILOR. Jack Mason had been to sea a great many times when I first knew him, and he has been a great many times since. He has sailed in a ship almost al…"
2,3,D0554,"White Jacket; Or, The World on a Man-of-War",0.1981,"… victualed, and appointed, and see how he acquits himself.” —FULLER’S “Good Sea-Captain.” NOTE. In the year 1843 I shipped as “ordinary seaman” on board of a United States frigate then lying in a…"
3,4,D0671,"The Life of Captain James Cook, the Circumnavigator",0.1858,"…ter, states in his Natural History of the Eastern Borders, that in 1692 the father of James Thomson, the author of The Seasons, was minister of Ednam, Roxburghshire, and a man named John Cook was one of the Elders of the Kirk. This John Cook ma…"
4,5,D0825,"The World of Waters, Or, A Peaceful Progress O'er the Unpathed Sea",0.1346,"…E] [Illustration: THE WORLD OF WATERS THE WORLD OF WATERS, OR, A Peaceful Progress o'er the Unpathed Sea. BY MRS. DAVID OSBORNE. With Illustrations. 1852. Contents CHAPTER I. The Wilton Family.-…"


... (6 outputs left out)

Q4: «ghost haunted castle mystery murder»

--- Ranking top-5: implementación MANUAL ---


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,D0948,Northumberland Yesterday and To-day,0.0832,"… and findeth the stream of a spring that ran with a great rushing, and nigh thereunto was a way that was much haunted. He abandoneth his high-way, and goeth all along the stream from the spring that lasteth a long league plenar…"
1,2,D0437,"Seeing Europe with Famous Authors, Volume 1",0.0631,…e ST. PAUL'S--By Augustus J.C. Hare THE BRITISH MUSEUM AND THE CRYSTAL PALACE--By H.A. Taine THE TEMPLE'S GALLERY OF GHOSTS PROM DICKENS--By J.R.G. Hassard THE TEMPLE CHURCH--By Augustus J.C. Hare LAMBETH CHURCH AND PALACE--By Augustus J.C. …
2,3,D0550,Miscellaneous Essays,0.0608,"… else might have seemed a heathen wilderness. This sort of religious talisman being secured, a man the most afraid of ghosts (like myself, suppose, or the reader) becomes armed into courage to wander for days in their sylvan recesses. The moun…"
3,4,D0780,"The Mirror of Literature, Amusement, and Instruction. Volume 13, No. 355, February 7, 1829",0.0575,"…rror in not having accepted the Alcoran; and instruct them that the dove which came down from heaven was not the Holy Ghost, but was Mahomet, who shall be again upon earth thirty years, and confirm the Alcoran by new miracles. After that time…"
4,5,D0405,The Old Man in the Corner,0.0502,"…on, the Countesses of Brockelsby having, seemingly, no predilection for twins. But in 1878 the mistress of Brockelsby Castle presented her lord with twin-born sons. ""Fortunately, in modern times, science is more wide-awake, and attendants …"


--- Ranking top-5: LIBRERÍA ---


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,D0948,Northumberland Yesterday and To-day,0.0832,"… and findeth the stream of a spring that ran with a great rushing, and nigh thereunto was a way that was much haunted. He abandoneth his high-way, and goeth all along the stream from the spring that lasteth a long league plenar…"
1,2,D0437,"Seeing Europe with Famous Authors, Volume 1",0.0631,…e ST. PAUL'S--By Augustus J.C. Hare THE BRITISH MUSEUM AND THE CRYSTAL PALACE--By H.A. Taine THE TEMPLE'S GALLERY OF GHOSTS PROM DICKENS--By J.R.G. Hassard THE TEMPLE CHURCH--By Augustus J.C. Hare LAMBETH CHURCH AND PALACE--By Augustus J.C. …
2,3,D0550,Miscellaneous Essays,0.0608,"… else might have seemed a heathen wilderness. This sort of religious talisman being secured, a man the most afraid of ghosts (like myself, suppose, or the reader) becomes armed into courage to wander for days in their sylvan recesses. The moun…"
3,4,D0780,"The Mirror of Literature, Amusement, and Instruction. Volume 13, No. 355, February 7, 1829",0.0575,"…rror in not having accepted the Alcoran; and instruct them that the dove which came down from heaven was not the Holy Ghost, but was Mahomet, who shall be again upon earth thirty years, and confirm the Alcoran by new miracles. After that time…"
4,5,D0405,The Old Man in the Corner,0.0502,"…on, the Countesses of Brockelsby having, seemingly, no predilection for twins. But in 1878 the mistress of Brockelsby Castle presented her lord with twin-born sons. ""Fortunately, in modern times, science is more wide-awake, and attendants …"


In [20]:
resumen = []
for qid, q in consultas.items():
    top_m = np.argsort(-q["sim_manual"], kind="stable")[:5]
    top_l = np.argsort(-q["sim_lib"], kind="stable")[:5]
    dif_top5 = np.abs(q["sim_manual"][top_m] - q["sim_lib"][top_l]).max()
    dif_total = np.abs(q["sim_manual"] - q["sim_lib"]).max()

    comp = pd.DataFrame({
        "Pos.": range(1, 6),
        "ID manual": [doc_ids[i] for i in top_m],
        "Score manual": q["sim_manual"][top_m],
        "ID librería": [doc_ids[i] for i in top_l],
        "Score librería": q["sim_lib"][top_l],
    })
    comp["|Diferencia|"] = (comp["Score manual"] - comp["Score librería"]).abs()
    print(f"\n{qid}: comparación lado a lado")
    display(comp)

    resumen.append({
        "Consulta": qid,
        "Dif. máx. top-5": dif_top5,
        "Dif. máx. todos los docs": dif_total,
        f"¿Dentro de tol ({TOLERANCIA})?": dif_total < TOLERANCIA,
        "¿Mismo top-5 (orden)?": list(top_m) == list(top_l),
    })

print("\nResumen de verificación")
display(pd.DataFrame(resumen))


Q1: comparación lado a lado


,Pos.,ID manual,Score manual,ID librería,Score librería,|Diferencia|
0,1,D0064,0.148977,D0064,0.148977,5.551115e-17
1,2,D0744,0.146552,D0744,0.146552,0.000000e+00
2,3,D0383,0.140529,D0383,0.140529,3.885781e-16
3,4,D0460,0.134679,D0460,0.134679,7.216450e-16
4,5,D0581,0.129963,D0581,0.129963,0.000000e+00



Q2: comparación lado a lado


,Pos.,ID manual,Score manual,ID librería,Score librería,|Diferencia|
0,1,D0439,0.287078,D0439,0.287078,9.992007e-16
1,2,D0928,0.237248,D0928,0.237248,5.551115e-17
2,3,D0554,0.198090,D0554,0.198090,6.383782e-16
3,4,D0671,0.185826,D0671,0.185826,9.714451e-16
4,5,D0825,0.134560,D0825,0.134560,6.661338e-16



Q3: comparación lado a lado


,Pos.,ID manual,Score manual,ID librería,Score librería,|Diferencia|
0,1,D0481,0.239102,D0481,0.239102,4.163336e-16
1,2,D0103,0.223073,D0103,0.223073,2.498002e-16
2,3,D0150,0.199447,D0150,0.199447,1.387779e-16
3,4,D0156,0.190102,D0156,0.190102,7.216450e-16
4,5,D0079,0.181367,D0079,0.181367,9.436896e-16



Q4: comparación lado a lado


,Pos.,ID manual,Score manual,ID librería,Score librería,|Diferencia|
0,1,D0948,0.083166,D0948,0.083166,1.082467e-15
1,2,D0437,0.063095,D0437,0.063095,3.191891e-16
2,3,D0550,0.060782,D0550,0.060782,1.665335e-16
3,4,D0780,0.057543,D0780,0.057543,5.551115e-17
4,5,D0405,0.050227,D0405,0.050227,1.110223e-16



Resumen de verificación


,Consulta,Dif. máx. top-5,Dif. máx. todos los docs,¿Dentro de tol (1e-09)?,¿Mismo top-5 (orden)?
0,Q1,7.216450e-16,7.514822e-15,True,True
1,Q2,9.992007e-16,4.080070e-15,True,True
2,Q3,9.436896e-16,2.664535e-15,True,True
3,Q4,1.082467e-15,1.082467e-15,True,True
